
# Algoritmo de Metropolis para o Modelo de Ising 2D

**Material de apoio ao TCC:** *Comparação e Otimização de Algoritmos de Monte Carlo
Aplicados ao Modelo de Ising* (Diego R. Oliveira, UFPR).

Este notebook implementa o algoritmo de **Metropolis** com dinâmica de inversão de
spin único (*single spin-flip*) para o modelo de Ising bidimensional, seguindo
fielmente a formulação apresentada nas Seções **2 (Modelo de Ising)**, **3.4
(Metropolis)** e **4.3.1 (Algoritmo de Metropolis)** do trabalho.

O objetivo é que este código sirva tanto para a validação e coleta de dados do TCC
quanto como **material didático para outros alunos da graduação** que queiram
entender, na prática, como uma simulação de Monte Carlo é construída do zero.

**Estrutura deste notebook:**
1. Representação da rede e estado inicial
2. Tabela de vizinhos (condições de contorno periódicas)
3. Cálculo de energia e magnetização (definições globais, usadas para validação)
4. Variação local de energia ($\Delta E$) e seus valores discretos
5. Tabela de busca (*lookup table*) para os fatores de Boltzmann
6. Critério de aceitação de Metropolis e um passo de atualização
7. Protocolo completo de simulação (equilibração + produção)
8. Testes de sanidade internos (conferência do código)
9. Validação preliminar contra a solução analítica de Onsager

Cada seção de código traz, em comentário, a equação correspondente do TCC (por
exemplo, `Eq. (4.6)`), para que o código possa ser lido lado a lado com o texto.

**Organização do código:** as funções que descrevem o *sistema físico* (rede,
vizinhos, energia, magnetização, solução de Onsager) são comuns aos três
algoritmos comparados no TCC e por isso vivem em um módulo compartilhado,
[`ising_utils.py`](https://github.com/SEU-USUARIO/ising-monte-carlo/blob/main/ising_utils.py),
importado logo abaixo. Apenas o que é *específico* do Metropolis (variação
local de energia, tabela de Boltzmann, critério de aceitação, protocolo de
simulação) é definido diretamente neste notebook.


In [ ]:

# Clona o repositório do projeto (contém o módulo compartilhado ising_utils.py)
# e adiciona ao caminho de importação do Python.

!git clone https://github.com/diegorafael1010/ising-monte-carlo.git
import sys
sys.path.append('/content/ising-monte-carlo')


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from ising_utils import (
    inicializar_rede,
    construir_tabela_vizinhos,
    energia_total,
    magnetizacao_total,
    temperatura_critica_onsager,
    energia_onsager,
    magnetizacao_onsager,
    J,
)

# Gerador de números aleatórios com semente fixa, para que os resultados deste
# notebook sejam reprodutíveis. Para rodadas de produção "de verdade", pode-se
# trocar a semente ou deixar aleatória (rng = np.random.default_rng()).
SEED = 42
rng = np.random.default_rng(SEED)


## Montagem do Google Drive

Antes de qualquer simulação, é preciso montar o Google Drive, já que é lá que os resultados (`.npz`) serão salvos, para persistirem entre sessões do Colab. A checagem abaixo confirma que a montagem funcionou de fato (não apenas criou uma pasta local com o mesmo nome, o que pode acontecer silenciosamente se a montagem falhar).

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

assert os.path.ismount('/content/drive'), (
    "ERRO: Google Drive não está montado de verdade! "
    "Não prossiga até resolver isso, veja as instruções acima."
)
print("Drive confirmado como montado corretamente.")


## 1. Representação da Rede e Estado Inicial

Seguindo a Seção 2.3 (Rede) e a Seção 4.3.1 do TCC, a rede quadrada de tamanho
linear $L$ é armazenada como um **arranjo unidimensional** de tamanho
$N = L^2$, com cada posição guardando o valor do spin $s_i \in \{-1, +1\}$.

Aqui usamos indexação **0-based** (padrão em Python), diferente da indexação
1-based usada na notação matemática do TCC (Equação 2.4). A relação entre a
posição de rede $(i, j)$ (linha, coluna) e o índice linear $k$ é:

$$k = i \cdot L + j, \qquad i, j \in \{0, 1, \dots, L-1\}.$$

Duas configurações iniciais são suportadas, como descrito no TCC:

- **Fria (`"fria"`)**: todos os spins iguais (estado ferromagnético ordenado,
  característico do limite $T \to 0$).
- **Quente (`"quente"`)**: cada spin sorteado independentemente (estado
  paramagnético desordenado, característico do limite $T \to \infty$).

A função `inicializar_rede` já foi importada de `ising_utils` na célula de
importações acima (é comum aos três algoritmos). O teste abaixo apenas
confirma seu comportamento.


In [ ]:

# Teste rápido: uma rede 4x4 fria deve ter todos os spins iguais a +1.
teste_fria = inicializar_rede(4, modo="fria")
assert np.all(teste_fria == 1), "Falha: rede fria deveria ter todos os spins +1"
print("Rede fria (L=4):", teste_fria)

# Uma rede quente deve, em média, ter magnetização próxima de zero para N grande.
teste_quente = inicializar_rede(64, modo="quente")
print("Magnetização média da rede quente (L=64):", teste_quente.mean(), "(esperado: perto de 0)")



## 2. Tabela de Vizinhos com Condições de Contorno Periódicas

Conforme a Seção 2.4 (Condições de Contorno) do TCC, adotam-se **condições de
contorno periódicas (PBC)**: o vizinho à direita do último sítio de uma linha é
o primeiro sítio da mesma linha, e de forma análoga para as demais direções.

Em vez de recalcular os vizinhos a cada passo (o que exigiria aritmética modular
repetida — `%`), pré-computamos uma **tabela de vizinhos**: para cada sítio $k$,
guardamos os índices lineares dos seus quatro vizinhos (norte, sul, leste,
oeste). Essa tabela é montada uma única vez, no início da simulação, e depois
apenas consultada — exatamente a otimização mencionada na Seção 2.3 do TCC
(*"tabelas de vizinhos alocadas na memória cache"*).

A função `construir_tabela_vizinhos` também já foi importada de
`ising_utils` (é idêntica para os três algoritmos, já que todos operam sobre
a mesma rede e a mesma Hamiltoniana de primeiros vizinhos).


In [ ]:

# Teste de sanidade da tabela de vizinhos: numa rede 3x3, o vizinho ao norte
# do sítio (0,0) [k=0] deve ser o sítio (2,0) [k=6], pela periodicidade.
viz_teste = construir_tabela_vizinhos(3)
assert viz_teste[0, 0] == 6, "Falha na periodicidade do vizinho norte"
print("Tabela de vizinhos para L=3, sítio k=0 (norte, sul, leste, oeste):", viz_teste[0])
print("OK: condições de contorno periódicas verificadas.")



## 3. Energia e Magnetização (Cálculo Global)

Estas funções calculam a energia total (Equação 2.6) e a magnetização total
(Equação 2.9) percorrendo **toda** a rede. Elas têm custo $\mathcal{O}(N)$ e
**não** são usadas a cada tentativa de flip (isso seria proibitivamente lento
— ver a discussão da Seção 4.3.1 sobre a vantagem de calcular $\Delta E$
localmente). Servem para:

- Definir a energia inicial da simulação;
- Validar, por conferência independente, que os cálculos incrementais de
  $\Delta E$ (Seção 4) estão corretos (Seção 8 deste notebook);
- Calcular as grandezas físicas apenas nos instantes de amostragem, durante a
  etapa de produção (bem mais raros que as tentativas de flip).

Adota-se a convenção de unidades reduzidas com $J = 1$ e $k_B = 1$, como
estabelecido na Seção 4.2 do TCC. As funções `energia_total` e
`magnetizacao_total` (assim como a constante `J`) já foram importadas de
`ising_utils`, pois são idênticas para os três algoritmos.


In [ ]:

# Teste de sanidade: numa rede 2x2 totalmente alinhada (fria), cada spin tem
# 4 vizinhos, mas em L=2 com PBC cada vizinho North/South/East/West aponta
# para os mesmos 2 outros sítios -- então usamos L=4 para um teste mais claro.
L_teste = 4
estado_teste = inicializar_rede(L_teste, modo="fria")
viz_teste = construir_tabela_vizinhos(L_teste)

E_teste = energia_total(estado_teste, viz_teste)
M_teste = magnetizacao_total(estado_teste)

# Para uma rede totalmente alinhada, todas as N*2 ligações contribuem com -J.
N_teste = L_teste ** 2
E_esperada = -J * 2 * N_teste  # 2 ligações "novas" por sítio (direita e baixo)
print(f"Energia total (rede fria, L={L_teste}): {E_teste} (esperado: {E_esperada})")
print(f"Magnetização total (rede fria, L={L_teste}): {M_teste} (esperado: {N_teste})")
assert E_teste == E_esperada
assert M_teste == N_teste
print("OK: energia e magnetização globais conferem com o valor esperado analiticamente.")



## 4. Variação Local de Energia ($\Delta E$)

Como discutido na Seção 4.3.1 (e na Seção 2.5) do TCC, a variação de energia
associada à tentativa de inversão de um único spin $s_k$ depende **apenas**
dos seus quatro primeiros vizinhos, e é dada pela Equação (4.6):

$$\Delta E_k = 2 J s_k \sum_{j \in nn(k)} s_j$$

Como cada $s_j \in \{-1, +1\}$ e há exatamente 4 vizinhos, a soma
$\sum_{j\in nn(k)} s_j$ só pode assumir os valores $\{-4, -2, 0, +2, +4\}$
(Equação 4.7), de modo que, com $J=1$, $\Delta E_k$ só pode assumir os cinco
valores discretos $\{-8, -4, 0, +4, +8\}$ (Equação 4.8). Isso é o que torna
possível pré-calcular os fatores de Boltzmann na Seção 5 a seguir.


In [ ]:

def delta_E_local(estado: list, vizinhos: list, k: int) -> float:
    """Calcula a variação de energia associada à inversão do spin k
    (Equação 4.6 do TCC), sem recalcular a energia de toda a rede.

    Nota de desempenho: `estado` e `vizinhos` aqui são listas Python, não
    arrays NumPy. Indexar um array NumPy com outro array ("fancy indexing"),
    como em `estado[vizinhos[k]]`, aloca um array temporário a cada chamada,
    um custo desprezível quando feito uma vez sobre a rede toda, mas que
    domina o tempo total quando repetido milhões de vezes, um sítio por vez,
    dentro do laço principal do Metropolis. Listas Python evitam esse custo
    para este tipo de acesso escalar.
    """
    n0, n1, n2, n3 = vizinhos[k]
    soma_vizinhos = estado[n0] + estado[n1] + estado[n2] + estado[n3]
    return 2.0 * J * estado[k] * soma_vizinhos


In [ ]:

# Teste de sanidade: o conjunto de valores possíveis de delta_E_local deve
# ser exatamente {-8, -4, 0, 4, 8}, para qualquer configuração (Equação 4.8).
L_teste = 16
estado_teste_np = inicializar_rede(L_teste, modo="quente")
viz_teste_np = construir_tabela_vizinhos(L_teste)

# delta_E_local agora opera sobre listas Python (ver nota de desempenho acima)
estado_teste = estado_teste_np.tolist()
vizinhos_teste = [tuple(int(x) for x in row) for row in viz_teste_np]

valores_encontrados = set()
for k in range(L_teste ** 2):
    valores_encontrados.add(delta_E_local(estado_teste, vizinhos_teste, k))

print("Valores de delta_E encontrados:", sorted(valores_encontrados))
assert valores_encontrados.issubset({-8.0, -4.0, 0.0, 4.0, 8.0})
print("OK: delta_E_local só assume os cinco valores previstos pela Equação 4.8.")



## 5. Tabela de Busca (*Lookup Table*) para os Fatores de Boltzmann

Como $\Delta E \le 0$ implica aceitação garantida ($P_{acc}=1$), só é
necessário avaliar o fator de Boltzmann $e^{-\beta \Delta E}$ para os dois
valores positivos possíveis, $\Delta E \in \{+4, +8\}$ (Equação 4.9 do TCC).
Pré-calculamos esses dois valores **uma única vez** por temperatura, no início
da simulação, evitando avaliar exponenciais repetidamente dentro do laço
principal do Metropolis.


In [ ]:

def construir_lookup_boltzmann(beta: float) -> dict:
    '''Pré-calcula os fatores de Boltzmann para as duas variações de energia
    positivas possíveis (Equação 4.9 do TCC).

    Parameters
    ----------
    beta : float
        Temperatura inversa, beta = 1/T (unidades reduzidas, k_B = 1).

    Returns
    -------
    dict
        Dicionário {4.0: exp(-beta*4), 8.0: exp(-beta*8)}.
    '''
    return {dE: np.exp(-beta * dE) for dE in (4.0, 8.0)}


In [ ]:

# Teste de sanidade: para beta muito grande (T -> 0), os fatores de Boltzmann
# devem ser próximos de zero (aceitar um aumento de energia fica muito raro).
lookup_frio = construir_lookup_boltzmann(beta=10.0)
print("Lookup table para T baixa (beta=10):", lookup_frio)
assert lookup_frio[8.0] < lookup_frio[4.0] < 0.1
print("OK: fatores de Boltzmann decrescem com o aumento de delta_E, como esperado.")



## 6. Critério de Aceitação de Metropolis e um Passo de Atualização

O critério de aceitação (Equação 3.14 do TCC) é:

$$A(\mu \to \nu) = \min\left(1, e^{-\beta \Delta E}\right)$$

ou seja: se $\Delta E \le 0$, a inversão é aceita imediatamente; caso
contrário, é aceita com probabilidade $e^{-\beta \Delta E}$ (consultada na
*lookup table* construída acima).

Um **passo de Monte Carlo por sítio (1 MCS/site)** corresponde a $N = L^2$
tentativas de inversão (Seção 4.3.1). Implementamos os dois esquemas de
seleção de sítio discutidos no TCC:

- `"aleatoria"`: o sítio é sorteado uniformemente a cada tentativa;
- `"sequencial"`: a rede é varrida em ordem, do sítio 0 ao $N-1$.

Ambos os esquemas produzem, no equilíbrio, as mesmas médias estatísticas
(Seção 4.3.1), mas a seleção sequencial tem melhor aproveitamento de cache.


In [ ]:

def tentativa_metropolis(
    estado: list,
    vizinhos: list,
    k: int,
    lookup: dict,
    rng: np.random.Generator,
) -> bool:
    """Executa uma única tentativa de inversão do spin k, seguindo o
    critério de Metropolis (Equação 3.14). Modifica `estado` in-place se a
    tentativa for aceita. `estado` e `vizinhos` são listas Python (ver nota
    de desempenho em delta_E_local).

    Returns
    -------
    bool
        True se a inversão foi aceita, False caso contrário.
    """
    dE = delta_E_local(estado, vizinhos, k)
    if dE <= 0:
        aceito = True
    else:
        r = rng.random()
        aceito = r < lookup[dE]

    if aceito:
        estado[k] *= -1

    return aceito


def sweep_metropolis(
    estado: list,
    vizinhos: list,
    lookup: dict,
    rng: np.random.Generator,
    esquema: str = "aleatoria",
) -> int:
    """Executa 1 MCS/site: N = L**2 tentativas de inversão (Seção 4.3.1).
    `estado` e `vizinhos` são listas Python (ver nota de desempenho em
    delta_E_local).

    Parameters
    ----------
    esquema : {"aleatoria", "sequencial"}
        Esquema de escolha do sítio a cada tentativa.

    Returns
    -------
    int
        Número de inversões aceitas nesse sweep (diagnóstico).
    """
    N = len(estado)
    if esquema == "aleatoria":
        sitios = rng.integers(0, N, size=N)
    elif esquema == "sequencial":
        sitios = range(N)
    else:
        raise ValueError("esquema deve ser 'aleatoria' ou 'sequencial'")

    aceitos = 0
    for k in sitios:
        if tentativa_metropolis(estado, vizinhos, int(k), lookup, rng):
            aceitos += 1
    return aceitos




In [ ]:

# Teste de sanidade: a beta = 0 (T infinita), TODA tentativa deve ser aceita,
# já que exp(-0*dE) = 1 sempre.
L_teste = 8
estado_teste = inicializar_rede(L_teste, modo="quente").tolist()
viz_teste = [tuple(int(x) for x in row) for row in construir_tabela_vizinhos(L_teste)]
lookup_infinito = construir_lookup_boltzmann(beta=0.0)

aceitos = sweep_metropolis(estado_teste, viz_teste, lookup_infinito, rng, esquema="aleatoria")
print(f"Aceitos em T=infinito: {aceitos} de {L_teste**2} tentativas (esperado: {L_teste**2})")
assert aceitos == L_teste ** 2
print("OK: a T infinita, todas as tentativas são aceitas, como esperado.")



## 7. Protocolo Completo de Simulação (Equilibração + Produção)

Reunindo as peças anteriores, esta função executa o protocolo descrito na
Seção 4.5.3 do TCC:

1. Inicializa a rede (fria ou quente);
2. Executa `n_eq` MCS/site de **equilíbração**, descartando todas as
   configurações geradas;
3. Executa `n_prod` MCS/site de **produção**, registrando energia e
   magnetização a cada `intervalo_amostragem` MCS/site.

Os valores de produção usados no TCC (Equações 4.25 e 4.26: $N_{eq}=10^5$,
$N_{prod}=10^6$) são grandes para fins de demonstração rápida neste notebook;
aqui usamos valores bem menores nos testes, e o valor final de produção só
deve ser usado nas rodadas oficiais de coleta de dados (mais lentas).


In [ ]:

def simular_metropolis(
    L: int,
    T: float,
    n_eq: int,
    n_prod: int,
    intervalo_amostragem: int = 10,
    modo_inicial: str = "quente",
    esquema: str = "aleatoria",
    rng: np.random.Generator = rng,
    mostrar_progresso: bool = False,
) -> dict:
    """Executa o protocolo completo de simulação de Metropolis para o
    modelo de Ising 2D (Seções 4.3.1 e 4.5.3 do TCC).

    Nota de desempenho: internamente, a rede e a tabela de vizinhos são
    convertidas para listas Python (ver nota em delta_E_local), a
    conversão de volta para NumPy só acontece nos instantes de amostragem,
    quando energia_total/magnetizacao_total (de ising_utils) são chamadas.

    Returns
    -------
    dict
        Ver versão anterior desta função.
    """
    N = L * L
    beta = 1.0 / T

    estado_np = inicializar_rede(L, modo=modo_inicial, rng=rng)
    vizinhos_np = construir_tabela_vizinhos(L)
    lookup = construir_lookup_boltzmann(beta)

    # Representação em listas Python, usada no laço quente das tentativas
    estado = estado_np.tolist()
    vizinhos = [tuple(int(x) for x in row) for row in vizinhos_np]

    iterador_eq = (
        tqdm(range(n_eq), desc=f"Equilibrando (L={L}, T={T})")
        if mostrar_progresso else range(n_eq)
    )
    for _ in iterador_eq:
        sweep_metropolis(estado, vizinhos, lookup, rng, esquema=esquema)

    energias = []
    magnetizacoes = []
    iterador_prod = (
        tqdm(range(n_prod), desc=f"Produção (L={L}, T={T})")
        if mostrar_progresso else range(n_prod)
    )
    for passo in iterador_prod:
        sweep_metropolis(estado, vizinhos, lookup, rng, esquema=esquema)
        if passo % intervalo_amostragem == 0:
            # Converte de volta para NumPy só aqui (raro, não é o laço quente)
            estado_np = np.array(estado, dtype=np.int8)
            e = energia_total(estado_np, vizinhos_np) / N
            m = abs(magnetizacao_total(estado_np)) / N
            energias.append(e)
            magnetizacoes.append(m)

    return {
        "energia_por_sitio": np.array(energias),
        "magnetizacao_abs_por_sitio": np.array(magnetizacoes),
        "L": L,
        "T": T,
        "n_eq": n_eq,
        "n_prod": n_prod,
    }


In [ ]:

# Teste rápido (parâmetros pequenos só para conferir que a função roda e que
# a energia parece razoável. NÃO é uma rodada de produção real).
resultado_teste = simular_metropolis(L=16, T=2.5, n_eq=200, n_prod=1000, intervalo_amostragem=5)
print("Energia média por sítio:", resultado_teste["energia_por_sitio"].mean())
print("Magnetização absoluta média por sítio:", resultado_teste["magnetizacao_abs_por_sitio"].mean())

plt.figure(figsize=(7, 3))
plt.plot(resultado_teste["energia_por_sitio"])
plt.xlabel("Amostra (a cada 5 MCS/site)")
plt.ylabel(r"Energia por sítio $e$")
plt.title(f"Traço de energia -- L={resultado_teste['L']}, T={resultado_teste['T']}")
plt.tight_layout()
plt.show()



## 8. Testes de Sanidade Internos

Antes de confiar no código para gerar dados do TCC, vale a pena conferir que
o cálculo **incremental** de $\Delta E$ (usado a cada tentativa, por ser
rápido) concorda com o cálculo **direto**, feito recalculando a energia total
antes e depois da inversão (mais lento, mas inequivocamente correto por
definição).

Isso é diferente dos testes de sanidade da Seção 4.4.3 do TCC (que
introduzem erros de propósito para verificar a sensibilidade do código a
vieses de implementação); aqui verificamos apenas a **corretude aritmética**
do cálculo incremental.


In [ ]:

def teste_consistencia_delta_E(L: int = 10, n_testes: int = 200, rng: np.random.Generator = rng) -> None:
    """Confere, para várias configurações e sítios aleatórios, que o
    delta_E incremental bate com a diferença de energia total calculada
    diretamente (antes e depois da inversão).
    """
    vizinhos_np = construir_tabela_vizinhos(L)
    vizinhos = [tuple(int(x) for x in row) for row in vizinhos_np]

    for _ in range(n_testes):
        estado_np = inicializar_rede(L, modo="quente", rng=rng)
        estado = estado_np.tolist()
        k = int(rng.integers(0, L * L))

        E_antes = energia_total(np.array(estado, dtype=np.int8), vizinhos_np)
        dE_incremental = delta_E_local(estado, vizinhos, k)

        estado[k] *= -1  # aplica a inversão manualmente
        E_depois = energia_total(np.array(estado, dtype=np.int8), vizinhos_np)
        dE_direto = E_depois - E_antes

        assert np.isclose(dE_incremental, dE_direto), (
            f"Inconsistência: incremental={dE_incremental}, direto={dE_direto}"
        )

    print(f"OK: delta_E incremental consistente com o cálculo direto em {n_testes} testes.")

teste_consistencia_delta_E()




## 9. Validação Preliminar contra a Solução Analítica de Onsager

Esta seção antecipa, de forma simplificada e rápida, o teste completo descrito
na Seção 4.4.1 do TCC: comparar a energia e a magnetização médias simuladas
com as curvas exatas de Onsager (Equações 4.21 e 4.22).

**Atenção:** esta é apenas uma checagem rápida de sanidade, com poucos MCS de
produção e uma única rede pequena — **não** substitui a validação completa e
estatisticamente rigorosa da Seção 4.4 do TCC, que deve ser feita com os
valores de produção definidos na Seção 4.5.3 ($N_{eq}=10^5$,
$N_{prod}=10^6$) e várias sementes/repetições.

Nota sobre a convenção do SciPy: a função `scipy.special.ellipk(m)` espera o
**parâmetro** $m = k_1^2$, não o módulo $k_1$ diretamente (Equação 4.23 do
TCC usa o módulo $k_1$) -- isso já está tratado dentro de `energia_onsager`,
em `ising_utils`. As três funções usadas abaixo
(`temperatura_critica_onsager`, `energia_onsager`, `magnetizacao_onsager`)
já foram importadas de lá, pois servirão também para validar Wolff e
Swendsen-Wang nos respectivos notebooks.


In [ ]:

# ATENÇÃO: valores pequenos só para demonstração rodar em poucos segundos.
# Para a validação oficial do TCC, é usado os parâmetros da Seção 4.5 (L maior,
# N_eq=1e5, N_prod=1e6) e comparados com barras de erro (método da Seção 4.7).

L_demo = 32
temperaturas_demo = np.array([1.5, 2.0, 2.269, 2.5, 3.0])

e_simulado = []
m_simulado = []
for T in temperaturas_demo:
    r = simular_metropolis(L=L_demo, T=T, n_eq=2000, n_prod=5000, intervalo_amostragem=5)
    e_simulado.append(r["energia_por_sitio"].mean())
    m_simulado.append(r["magnetizacao_abs_por_sitio"].mean())

e_simulado = np.array(e_simulado)
m_simulado = np.array(m_simulado)

temperaturas_finas = np.linspace(1.2, 3.5, 200)
e_exato = np.array([energia_onsager(T) for T in temperaturas_finas])
m_exato = np.array([magnetizacao_onsager(T) for T in temperaturas_finas])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(temperaturas_finas, e_exato, "-", color="black", label="Onsager (exato, $L\\to\\infty$)")
ax1.plot(temperaturas_demo, e_simulado, "o", color="crimson", label=f"Metropolis (L={L_demo}, demo rápida)")
ax1.axvline(temperatura_critica_onsager(), ls="--", color="gray", lw=1, label="$T_c$")
ax1.set_xlabel("Temperatura $T$")
ax1.set_ylabel("Energia por sítio $e$")
ax1.legend()

ax2.plot(temperaturas_finas, m_exato, "-", color="black", label="Onsager (exato, $L\\to\\infty$)")
ax2.plot(temperaturas_demo, m_simulado, "o", color="crimson", label=f"Metropolis (L={L_demo}, demo rápida)")
ax2.axvline(temperatura_critica_onsager(), ls="--", color="gray", lw=1, label="$T_c$")
ax2.set_xlabel("Temperatura $T$")
ax2.set_ylabel("Magnetização absoluta por sítio $|m|$")
ax2.legend()

plt.suptitle("Validação preliminar (demonstração rápida -- não é a validação oficial do TCC)")
plt.tight_layout()
plt.show()


# **SIMULAÇÕES**


## Simulações Oficiais de Produção

A partir daqui, executam-se as simulações oficiais do Metropolis usadas nos
resultados do TCC (Seção 4.5.3 da Metodologia), com $N_{eq}=10^5$ e
$N_{prod}=10^6$ MCS/site. A grade de combinações de $L$ e $T$ segue a
planilha de controle (`controle_simulacoes_diego.xlsx`, cópia pessoal salva
no Drive).




### Por que salvar os dados, em que formato, e para quê depois

Cada rodada de simulação produz três tipos de informação que serão usados
em capítulos diferentes do TCC, então tudo é salvo junto, no mesmo arquivo:

- **`energia`** e **`magnetizacao`**: as séries temporais completas (um
  valor por amostra, não só a média) medidas durante a etapa de produção.
  São necessárias na íntegra (não apenas a média), porque o notebook de
  autocorrelação (Seção 4.7 do TCC) precisa da sequência completa de
  medidas para calcular $\tau_{\text{int}}$ via FFT (a autocorrelação é
  calculada *entre* pontos da série, então não há como reconstruí-la a
  partir de uma média já calculada).
- **`tempo_total_segundos`** e **`tempo_por_passo`**: usados na análise de
  desempenho computacional (Seção 4.8 do TCC), para calcular o Fator de
  Mérito de cada algoritmo.

**Formato escolhido: `.npz`** (arquivo comprimido do NumPy, criado com
`np.savez`). Foi escolhido em vez de, por exemplo, `.csv`, porque guarda
múltiplos arrays (de tamanhos diferentes) e números soltos no mesmo
arquivo, sem precisar de conversões, e porque o NumPy o lê de volta
rapidamente, já no formato de array pronto para uso, não sendo necessário
reprocessar todo o texto.

**Cada arquivo é salvo direto no Google Drive**, e não apenas na memória do
Colab, porque o ambiente do Colab é temporário: ao encerrar ou reiniciar a
sessão, tudo que estiver só na memória (incluindo a variável `resultado`)
é perdido. O Drive persiste entre sessões, então mesmo rodando as 60
simulações ao longo de vários dias, em sessões separadas do Colab, nada se
perde.

**Como os arquivos serão reabertos depois**, no notebook de autocorrelação
e no de desempenho:

```python
dados = np.load('/content/drive/MyDrive/0_TCC_Diego/dados/metropolis_L64_T2.269.npz')
energia = dados['energia']
magnetizacao = dados['magnetizacao']
tempo_total = dados['tempo_total_segundos']
```

Ou seja: **esta etapa (rodar e salvar) só precisa ser feita uma vez por
combinação de $(L, T)$**. Depois, a partir desses arquivos `.npz` já salvosserão usados para gerar gráficos e/ou tabelas.


## Executando uma combinação por vez (recomendado)

A célula abaixo roda **uma única combinação de $(L, T)$** por execução, com
barra de progresso (via `tqdm`) para acompanhamento, importante para as
rodadas maiores ($L=64$, $L=128$), que podem levar de vários minutos a
cerca de meia hora. Edite os valores de `L` e `T` conforme a linha da
planilha antes de rodar cada vez, e marque a linha correspondente como
"Concluído" na planilha ao final.

In [ ]:
import time
import os

# Reforço: garante que o Drive está montado antes de salvar (caso esta
# célula seja executada isoladamente, numa sessão nova, sem rodar a
# montagem lá do início do notebook)
assert os.path.ismount('/content/drive'), "ERRO: Google Drive não está montado! Rode a célula de montagem primeiro."

# ==== EDITE AQUI a cada rodada, conforme a linha da planilha ====
L = 128
T = 2.5
# ==================================================================

# O caminho da pasta no googledrive pode ser alterado conforme a necessidade.
pasta_saida = '/content/drive/MyDrive/0_TCC_Diego/dados'
os.makedirs(pasta_saida, exist_ok=True)
nome_arquivo = f'{pasta_saida}/metropolis_L{L}_T{T}.npz'

if os.path.exists(nome_arquivo):
    print(f"Já existe um arquivo salvo para esta combinação:\n  {nome_arquivo}")
    print("Apague o arquivo antigo (ou edite o nome acima) se quiser rodar de novo.")
else:
    tempo_inicio = time.perf_counter()
    resultado = simular_metropolis(
        L=L, T=T, n_eq=100_000, n_prod=1_000_000,
        intervalo_amostragem=10, mostrar_progresso=True,
    )
    tempo_fim = time.perf_counter()

    tempo_total = tempo_fim - tempo_inicio
    tempo_por_passo = tempo_total / (100_000 + 1_000_000)

    np.savez(
        nome_arquivo,
        energia=resultado['energia_por_sitio'],
        magnetizacao=resultado['magnetizacao_abs_por_sitio'],
        tempo_total_segundos=tempo_total,
        tempo_por_passo=tempo_por_passo,
    )

    # Confirma de verdade que o arquivo foi salvo, em vez de confiar só no print
    assert os.path.exists(nome_arquivo), "ERRO: o arquivo não foi salvo! Verifique o Drive."
    tamanho_kb = os.path.getsize(nome_arquivo) / 1024

    print(f"\nConcluído: L={L}, T={T}")
    print(f"Tempo total: {tempo_total:.1f}s ({tempo_total/60:.1f} min)")
    print(f"Energia média: {resultado['energia_por_sitio'].mean():.5f}")
    print(f"Magnetização média: {resultado['magnetizacao_abs_por_sitio'].mean():.5f}")
    print(f"Salvo em: {nome_arquivo} ({tamanho_kb:.1f} KB)")

### (Opcional) Executando várias combinações em sequência

A célula abaixo roda **todas** as combinações de $L$ e $T$ automaticamente,
uma após a outra, pulando as que já tiverem arquivo salvo. Útil se for necessário deixar o código rodando sem supervisão. Mas é preciso observar que o tempo total estimado para conclusão, pode chegar a várias horas para a grade completa e que também existe o risco de desconexão do Colab (versão gratuita) em sessões muito longas.

In [ ]:
import time
import os

# Reforço: garante que o Drive está montado antes de rodar qualquer coisa
assert os.path.ismount('/content/drive'), "ERRO: Google Drive não está montado! Rode a célula de montagem primeiro."

L_valores = [16, 32, 64, 128]
T_valores = [1.50, 2.00, 2.269, 2.50, 3.00]
pasta_saida = '/content/drive/MyDrive/0_TCC_Diego/dados'
os.makedirs(pasta_saida, exist_ok=True)

for L in L_valores:
    for T in T_valores:
        nome_arquivo = f'{pasta_saida}/metropolis_L{L}_T{T}.npz'
        if os.path.exists(nome_arquivo):
            print(f"Já existe: L={L}, T={T} -- pulando.")
            continue

        print(f"Rodando Metropolis L={L}, T={T}...")
        tempo_inicio = time.perf_counter()
        resultado = simular_metropolis(
            L=L, T=T, n_eq=100_000, n_prod=1_000_000,
            intervalo_amostragem=10, mostrar_progresso=True,
        )
        tempo_fim = time.perf_counter()

        tempo_total = tempo_fim - tempo_inicio
        tempo_por_passo = tempo_total / (100_000 + 1_000_000)

        np.savez(
            nome_arquivo,
            energia=resultado['energia_por_sitio'],
            magnetizacao=resultado['magnetizacao_abs_por_sitio'],
            tempo_total_segundos=tempo_total,
            tempo_por_passo=tempo_por_passo,
        )

        # Confirma de verdade que salvou, não confia só no print
        assert os.path.exists(nome_arquivo), f"ERRO: falha ao salvar {nome_arquivo}!"
        print(f"  Concluído em {tempo_total:.1f}s -- salvo e confirmado.\n")

## Script para testar dados já rodados

In [ ]:
"""
graficos_resultados.py
=======================

Carrega os arquivos .npz salvos pelas simulações oficiais (Seção 4.5.3 do
TCC) e gera gráficos de acompanhamento: energia e magnetização por sítio
comparadas com a solução exata de Onsager, para os tamanhos de rede já
simulados, além de um gráfico opcional de tempo de execução por L.
"""

import glob
import os
import re

import numpy as np
import matplotlib.pyplot as plt

from ising_utils import energia_onsager, magnetizacao_onsager, temperatura_critica_onsager

# ---------------------------------------------------------------------------
# Configuração -- ajuste conforme necessário
# ---------------------------------------------------------------------------
PASTA_DADOS = '/content/drive/MyDrive/0_TCC_Diego/dados'
ALGORITMO = 'metropolis'  # 'metropolis', 'wolff' ou 'swendsen_wang'

# ---------------------------------------------------------------------------
# 1. Descobre automaticamente quais arquivos já existem
# ---------------------------------------------------------------------------
padrao = os.path.join(PASTA_DADOS, f'{ALGORITMO}_L*_T*.npz')
arquivos = sorted(glob.glob(padrao))

if not arquivos:
    raise FileNotFoundError(
        f"Nenhum arquivo encontrado em {padrao}. Verifique PASTA_DADOS e ALGORITMO."
    )

regex_nome = re.compile(rf'{ALGORITMO}_L(\d+)_T([\d.]+)\.npz$')

dados_por_L = {}  # {L: {"T": [...], "e_media": [...], "e_erro": [...], "m_media": [...], "m_erro": [...], "tempo": [...]}}

for caminho in arquivos:
    nome = os.path.basename(caminho)
    m = regex_nome.search(nome)
    if not m:
        print(f"Aviso: nome de arquivo não reconhecido, ignorando: {nome}")
        continue

    L = int(m.group(1))
    T = float(m.group(2))

    dados = np.load(caminho)
    energia = dados['energia']
    magnetizacao = dados['magnetizacao']
    tempo_total = float(dados['tempo_total_segundos'])

    # Erro "ingênuo" (Equação 4.24-ish do TCC, sem correção de autocorrelação
    # -- SUBESTIMA a incerteza real; usado aqui apenas para referência visual
    # rápida. A barra de erro estatisticamente correta será calculada no
    # notebook de autocorrelação, Seção 4.7 do TCC.)
    e_erro_ingenuo = energia.std(ddof=1) / np.sqrt(len(energia))
    m_erro_ingenuo = magnetizacao.std(ddof=1) / np.sqrt(len(magnetizacao))

    dados_por_L.setdefault(L, {"T": [], "e_media": [], "e_erro": [], "m_media": [], "m_erro": [], "tempo": []})
    dados_por_L[L]["T"].append(T)
    dados_por_L[L]["e_media"].append(energia.mean())
    dados_por_L[L]["e_erro"].append(e_erro_ingenuo)
    dados_por_L[L]["m_media"].append(magnetizacao.mean())
    dados_por_L[L]["m_erro"].append(m_erro_ingenuo)
    dados_por_L[L]["tempo"].append(tempo_total)

# Ordena por temperatura dentro de cada L
for L in dados_por_L:
    ordem = np.argsort(dados_por_L[L]["T"])
    for chave in ("T", "e_media", "e_erro", "m_media", "m_erro", "tempo"):
        dados_por_L[L][chave] = np.array(dados_por_L[L][chave])[ordem]

L_valores_encontrados = sorted(dados_por_L.keys())
print(f"Tamanhos de rede encontrados para '{ALGORITMO}': {L_valores_encontrados}")

# ---------------------------------------------------------------------------
# 2. Gráfico principal: energia e magnetização vs T, comparado com Onsager
# ---------------------------------------------------------------------------
cores = plt.cm.viridis(np.linspace(0.15, 0.85, len(L_valores_encontrados)))

T_fino = np.linspace(1.2, 3.5, 300)
e_exato = np.array([energia_onsager(T) for T in T_fino])
m_exato = np.array([magnetizacao_onsager(T) for T in T_fino])
Tc = temperatura_critica_onsager()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(T_fino, e_exato, "-", color="black", lw=1.5, label="Onsager (exato, $L\\to\\infty$)")
for L, cor in zip(L_valores_encontrados, cores):
    d = dados_por_L[L]
    ax1.errorbar(d["T"], d["e_media"], yerr=d["e_erro"], fmt="o", color=cor,
                 capsize=3, label=f"$L={L}$")
ax1.axvline(Tc, ls="--", color="gray", lw=1, label="$T_c$")
ax1.set_xlabel("Temperatura $T$")
ax1.set_ylabel("Energia por sítio $e$")
ax1.set_title(f"Energia -- {ALGORITMO}")
ax1.legend(fontsize=9)

ax2.plot(T_fino, m_exato, "-", color="black", lw=1.5, label="Onsager (exato, $L\\to\\infty$)")
for L, cor in zip(L_valores_encontrados, cores):
    d = dados_por_L[L]
    ax2.errorbar(d["T"], d["m_media"], yerr=d["m_erro"], fmt="o", color=cor,
                 capsize=3, label=f"$L={L}$")
ax2.axvline(Tc, ls="--", color="gray", lw=1, label="$T_c$")
ax2.set_xlabel("Temperatura $T$")
ax2.set_ylabel("Magnetização absoluta por sítio $|m|$")
ax2.set_title(f"Magnetização -- {ALGORITMO}")
ax2.legend(fontsize=9)

plt.suptitle(
    "Acompanhamento dos dados salvos -- barras de erro são a estimativa "
    "ingênua (sem correção de autocorrelação, Seção 4.7 do TCC)",
    fontsize=10,
)
plt.tight_layout()
plt.savefig(os.path.join(PASTA_DADOS, f'grafico_{ALGORITMO}_energia_magnetizacao.png'), dpi=150)
plt.show()

# ---------------------------------------------------------------------------
# 3. Gráfico bônus: tempo de execução por L (visão preliminar de desempenho)
# ---------------------------------------------------------------------------
fig2, ax3 = plt.subplots(figsize=(6, 4))
for L, cor in zip(L_valores_encontrados, cores):
    d = dados_por_L[L]
    ax3.plot(d["T"], np.array(d["tempo"]) / 60, "o-", color=cor, label=f"$L={L}$")
ax3.axvline(Tc, ls="--", color="gray", lw=1, label="$T_c$")
ax3.set_xlabel("Temperatura $T$")
ax3.set_ylabel("Tempo total de execução (min)")
ax3.set_title(f"Tempo de execução por rodada -- {ALGORITMO}\n(visão preliminar; análise formal na Seção 4.8)")
ax3.legend(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(PASTA_DADOS, f'grafico_{ALGORITMO}_tempo.png'), dpi=150)
plt.show()

# ---------------------------------------------------------------------------
# 4. Resumo numérico em texto (para conferência rápida)
# ---------------------------------------------------------------------------
print("\nResumo:")
for L in L_valores_encontrados:
    d = dados_por_L[L]
    print(f"\nL={L}:")
    for i in range(len(d["T"])):
        print(
            f"  T={d['T'][i]:.3f}  e={d['e_media'][i]:.5f}±{d['e_erro'][i]:.5f}  "
            f"m={d['m_media'][i]:.5f}±{d['m_erro'][i]:.5f}  "
            f"tempo={d['tempo'][i]/60:.1f} min"
        )

# **Resultados do Metropolis (dados após as 20 simulações)**

Com as 20 combinações de $(L, T)$ concluídas (Seção 4.5 do TCC), esta seção
consolida os resultados: carrega todos os arquivos `.npz` salvos e compara
a energia e a magnetização médias com a solução exata de Onsager, agora
com a grade completa e oficial de tamanhos de rede.

**Escopo desta seção**: valida apenas o Metropolis isoladamente (Seção
4.4.1 do TCC). A comparação cruzada entre os três algoritmos (Seção 4.4.2)
só será possível depois que Wolff e Swendsen-Wang também estiverem
concluídos. A análise de autocorrelação (Seção 4.7) e desempenho (Seção
4.8) será feita em notebooks dedicados, compartilhados entre os três
algoritmos.

**Sobre as barras de erro do gráfico abaixo:** elas são calculadas pela
fórmula usual, desvio padrão dividido pela raiz do número de amostras.
Essa conta parte do princípio de que as amostras são independentes entre
si, o que não é bem verdade aqui, já que medidas consecutivas da mesma
simulação são correlacionadas (Como descrito na Seção 4.7 do TCC).
Por isso, essas barras tendem a ser menores do que o erro real. Elas
servem bem para uma conferência **visual rápida** agora, mas não devem ser
usadas como as incertezas finais, isso será recalculado no
notebook de autocorrelação, levando em conta o tempo de autocorrelação
$\tau_{\text{int}}$ de cada medida.

In [ ]:
import glob
import re

from ising_utils import energia_onsager, magnetizacao_onsager, temperatura_critica_onsager

PASTA_DADOS = '/content/drive/MyDrive/0_TCC_Diego/dados'
ALGORITMO = 'metropolis'

padrao = os.path.join(PASTA_DADOS, f'{ALGORITMO}_L*_T*.npz')
arquivos = sorted(glob.glob(padrao))
regex_nome = re.compile(rf'{ALGORITMO}_L(\d+)_T([\d.]+)\.npz$')

dados_por_L = {}
for caminho in arquivos:
    m = regex_nome.search(os.path.basename(caminho))
    if not m:
        continue
    L, T = int(m.group(1)), float(m.group(2))
    dados = np.load(caminho)
    energia, magnetizacao = dados['energia'], dados['magnetizacao']

    e_erro = energia.std(ddof=1) / np.sqrt(len(energia))
    m_erro = magnetizacao.std(ddof=1) / np.sqrt(len(magnetizacao))

    dados_por_L.setdefault(L, {"T": [], "e": [], "e_erro": [], "m": [], "m_erro": []})
    dados_por_L[L]["T"].append(T)
    dados_por_L[L]["e"].append(energia.mean())
    dados_por_L[L]["e_erro"].append(e_erro)
    dados_por_L[L]["m"].append(magnetizacao.mean())
    dados_por_L[L]["m_erro"].append(m_erro)

for L in dados_por_L:
    ordem = np.argsort(dados_por_L[L]["T"])
    for chave in dados_por_L[L]:
        dados_por_L[L][chave] = np.array(dados_por_L[L][chave])[ordem]

print(f"Tamanhos de rede encontrados: {sorted(dados_por_L.keys())}")
print(f"Total de arquivos carregados: {len(arquivos)} (esperado: 20)")

# --- Gráfico de validação (Seção 4.4.1) ---
cores = plt.cm.viridis(np.linspace(0.15, 0.85, len(dados_por_L)))
T_fino = np.linspace(1.2, 3.5, 300)
e_exato = np.array([energia_onsager(T) for T in T_fino])
m_exato = np.array([magnetizacao_onsager(T) for T in T_fino])
Tc = temperatura_critica_onsager()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.plot(T_fino, e_exato, "-", color="black", label="Onsager (exato)")
ax2.plot(T_fino, m_exato, "-", color="black", label="Onsager (exato)")

for (L, d), cor in zip(sorted(dados_por_L.items()), cores):
    ax1.errorbar(d["T"], d["e"], yerr=d["e_erro"], fmt="o", color=cor, capsize=3, label=f"$L={L}$")
    ax2.errorbar(d["T"], d["m"], yerr=d["m_erro"], fmt="o", color=cor, capsize=3, label=f"$L={L}$")

for ax, ylabel in ((ax1, "Energia por sítio $e$"), (ax2, "Magnetização absoluta $|m|$")):
    ax.axvline(Tc, ls="--", color="gray", lw=1, label="$T_c$")
    ax.set_xlabel("Temperatura $T$")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=9)

plt.suptitle("Validação do Metropolis contra a solução de Onsager (Seção 4.4.1)")
plt.tight_layout()
plt.savefig(os.path.join(PASTA_DADOS, 'resultados_metropolis.png'), dpi=150)
plt.show()